In [ ]:
#!pip install openai

In [ ]:
import json
import re
import os
from collections import OrderedDict
from dotenv import load_dotenv
from openai import OpenAI

JSON_STEP1_FOLDER = f"output/json/step_1/"
JSON_STEP2_FOLDER = f"output/json/step_2/"

In [2]:
OPENAI_MODEL = "gpt-4o-mini" #gpt-5-mini

In [3]:
# ner_prompt_config.py
# -*- coding: utf-8 -*-
"""
Configurazione prompt per LLM/NER:
- SYSTEM_PROMPT
- USER_INSTRUCTIONS
- JSON_SCHEMA_STR
- Helper build_user_message()
- EXAMPLE_MESSAGES
"""

'\nConfigurazione prompt per LLM/NER:\n- SYSTEM_PROMPT\n- USER_INSTRUCTIONS\n- JSON_SCHEMA_STR\n- Helper build_user_message()\n- EXAMPLE_MESSAGES\n'

In [4]:
#
# libero -- (vincolo assoluto)
#
# =========================
# 1) SYSTEM PROMPT
# =========================
SYSTEM_PROMPT = r"""
Sei un agente specializzato in estrazione e arricchimento di entità da testi istituzionali italiani (protocolli, convenzioni, atti, delibere).
Produci esclusivamente JSON valido secondo lo schema indicato, senza testo extra.

Obiettivo
---------
Per ogni entità individuata nel testo, estrai le chiavi:
- nome, tipo, comune, indirizzo, ente_capofila, note

e arricchisci con:
- provincia, regione (se ricavabili)

Aggiungi metadati facoltativi:
- fonte_indirizzo, fonte_provincia, fonte_regione ∈ {testo, dedotto, conoscenza}
- conflitti (solo se presenti)
- confidence ∈ [0.0, 1.0]

Ontologia e normalizzazione
---------------------------
1) tipo ∈ {
   "Ente territoriale",
   "ASL",
   "Consorzio socio-assistenziale",
   "Associazione di volontariato (OdV)",
   "Associazione di promozione sociale (APS)",
   "Altro"
}
2) nome: mantieni grafia ufficiale; sigle in MAIUSCOLO (ASL, ODV, APS, ONLUS). Pulisci spazi doppi/virgolette ornamentali.
3) indirizzo (valido se contiene toponimo + nome via/piazza e preferibilmente civico).
   Toponimi ammessi (case-insensitive): Via|Viale|V\.le|Vicolo|Corso|C\.so|Piazza|P\.zza|Largo|Piazzale|P\.le|Strada|Borgo|Traversa|Località|Frazione|Regione.
   Accetta CAP (\b\d{5}\b) e sigla provincia tra parentesi (es. (BI)).
4) Comune/Provincia/Regione:
   - Se presenti nell’indirizzo → estrai.
   - Se assenti ma il comune è noto → deduci provincia e regione dal comune.
   - Se trovi solo sigla provincia → valorizza provincia se noto altrimenti conserva sigla e riduci confidence.

Pipeline (ordine obbligatorio)
------------------------------
1) Preprocessa: pulisci whitespace e apostrofi; unisci righe spezzate di un indirizzo.
2) Estrai entità: crea un record per ogni organizzazione (Comune di…, Provincia di…, Ambito…, ASL…, Consorzio…, Associazione…, Procura…, Università…, Dipartimento…, Cooperativa…).
3) Gestione indirizzo:
   Caso 1 — indirizzo vuoto:
     1a) Se comune è valorizzato → lascia quel comune.
     1b) Altrimenti, se nome è valorizzato → cerca comune valido per quel nome nel contesto.
   Caso 2 — indirizzo presente:
     - Estrai/deduci comune dall’indirizzo; se diverso dal campo comune, registra conflitto.
     - Estrai/deduci provincia e regione.
4) Fonti:
   - Dal testo → "testo"
   - Deduci da altro campo → "dedotto"
   - Da conoscenza esterna → "conoscenza"
5) Conflitti:
   - Se comune (record) ≠ comune (da indirizzo) → conflitti.comune = { "dato": "...", "trovato": "..." }
6) De-duplicazione:
   - Duplicato = stesso nome + stesso comune
   - Mantieni l’indirizzo più specifico, sposta alternative in note.
7) Confidence:
   - Base 0.50 se nome+tipo trovati
   - +0.15 indirizzo con civico; +0.10 senza civico ma univoco
   - +0.10 coerenza comune–provincia–regione
   - −0.20 conflitto comune; −0.10 provincia/regione parziali
   - Clampa a [0.0, 1.0]

Output (solo JSON)
------------------
{
  "file": "<nome_file_input_o_placeholder>",
  "entities": [
    {
      "nome": "",
      "tipo": "",
      "comune": "",
      "indirizzo": "",
      "ente_capofila": "",
      "note": "",
      "provincia": "",
      "regione": "",
      "fonte_indirizzo": "testo|dedotto|conoscenza",
      "fonte_provincia": "testo|dedotto|conoscenza",
      "fonte_regione": "testo|dedotto|conoscenza",
      "conflitti": {
        "comune": { "dato": "", "trovato": "" }
      },
      "confidence": 0.0
    }
  ]
}
""".strip()

# =============================
# 2) USER INSTRUCTIONS
# =============================
USER_INSTRUCTIONS = r"""
Analizza il testo fornito ed estrai per ogni entità le chiavi:
- nome, tipo, comune, indirizzo, ente_capofila, note

Poi applica rigorosamente:

1) Se "indirizzo" è vuoto:
   a) Se comune è vuoto → cerca un comune valido per quel nome.
   b) Se "nome" è valorizzato → cerca un comune valido per quel nome. 
   In entrambi i casi → valorizza anche provincia e regione (dedotte se necessario).

2) Se "indirizzo" è valorizzato:
   - Estrai/deduci "comune"; se differisce → registra conflitto.
   - Valorizza "provincia" e "regione" (o deducile dal comune).
   - Non cambiarlo.

3) Normalizza:
   - Pulisci spazi/virgolette; sigle in MAIUSCOLO.
   - Uniforma "tipo" alle categorie predefinite. 
   - Normalizza sempre provincia al nome esteso (es. Cuneo) e, se disponibile, aggiungi anche provincia_sigla (es. CN). Non usare solo la sigla in provincia.
  -  Non impostare confidence a 1.0 se manca il CAP o se provincia/regione sono dedotte; in tali casi, max 0.95.
  -  file in output deve essere identico al nome file dichiarato nel messaggio utente.

4) De-duplicazione:
   - Stesso nome+comune → unisci; tieni indirizzo più specifico; altri in note.

5) Provenienza e qualità:
   - Fonte dal testo → "testo"
   - Fonte dedotta → "dedotto"
   - Fonte da conoscenza → "conoscenza"
   - Conflitti nel campo "conflitti"
   - Aggiungi "confidence" ∈ [0.0, 1.0]

6) Output:
   - Solo JSON valido come nello schema del SYSTEM_PROMPT.
""".strip()

# =========================
# 3) JSON Schema (opzionale)
# =========================
JSON_SCHEMA_STR = r"""
{
  "type": "object",
  "required": ["file", "entities"],
  "properties": {
    "file": { "type": "string" },
    "entities": {
      "type": "array",
      "items": {
        "type": "object",
        "required": ["nome", "tipo", "comune", "indirizzo", "ente_capofila", "note"],
        "properties": {
          "nome": { "type": "string" },
          "tipo": { "type": "string" },
          "comune": { "type": "string" },
          "indirizzo": { "type": "string" },
          "ente_capofila": { "type": "string" },
          "note": { "type": "string" },
          "provincia": { "type": "string" },
          "regione": { "type": "string" },
          "fonte_indirizzo": { "type": "string" },
          "fonte_provincia": { "type": "string" },
          "fonte_regione": { "type": "string" },
          "conflitti": { "type": "object" },
          "confidence": { "type": "number" }
        }
      }
    }
  }
}
""".strip()

In [5]:
# Chiave API
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [6]:
# 🛡️ Funzione difensiva per estrarre entities da item
def safe_get_entities(item: dict):
    """
    Estrae l'elenco 'entities' da un record JSON
    gestendo i casi in cui 'risultato' sia None o non sia un dict.
    """
    if not isinstance(item, dict):
        return []
    r = item.get("risultato") or {}
    if not isinstance(r, dict):
        return []
    ents = r.get("entities") or []
    return ents if isinstance(ents, list) else []

In [ ]:
def extract_json_from_text(testo: str):
    
    if testo is None:
        return None, "output_text è None"
    m = re.search(r"```json\s*(\{.*?\})\s*```", testo, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1)), None
        except Exception as e:
            return None, f"JSONDecodeError in blocco ```json```: {e}"
    m = re.search(r"```\s*(\{.*?\})\s*```", testo, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1)), None
        except Exception:
            pass
    start = testo.find('{'); end = testo.rfind('}')
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(testo[start:end+1]), None
        except Exception as e:
            return None, f"JSONDecodeError nel fallback: {e}"
    return None, "Nessun oggetto JSON trovato nel testo" 

In [ ]:
def save_result(output_finale, output_file):
    """
    Salva i risultati in formato JSON in una cartella di output.

    Args:
        output_finale (list|dict): struttura dati da serializzare in JSON.
        output_file (str): cartella di output + nome del file JSON di destinazione.
        

    Returns:
        str: percorso completo del file salvato.
    """
    

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(output_finale, f, indent=4, ensure_ascii=False)

    print(f"✅ Risultati salvati in: {output_file}")
    return output_file



In [9]:
# Helper per creare il messaggio utente new
def build_user_message(entity: dict, file_txt: str) -> str:
    # Estrai codice (prime due cifre) e regione
    codice = (file_txt.split("_", 1)[0] or "").strip()
    istat_to_regione = {
        "01":"Piemonte","02":"Valle D'Aosta","03":"Lombardia","04":"Trentino Alto Adige","05":"Veneto",
        "06":"Friuli Venezia Giulia","07":"Liguria","08":"Emilia-Romagna","09":"Toscana","10":"Umbria",
        "11":"Marche","12":"Lazio","13":"Abruzzo","14":"Molise","15":"Campania","16":"Puglia",
        "17":"Basilicata","18":"Calabria","19":"Sicilia","20":"Sardegna"
    }
    regione_file = istat_to_regione.get(codice, "")

    ent_str = json.dumps(entity, ensure_ascii=False)
    return (
        f"FILE: {file_txt}\n"
        f"CODICE_ISTAT_REGIONE: {codice}\n"
        f"REGIONE_DA_FILE: {regione_file}\n\n"
        f"ISTRUZIONI:\n{USER_INSTRUCTIONS}\n\n"
        f"ENTITY DI PARTENZA:\n{ent_str}\n---"
    )
  
  

In [ ]:

def main(start_reg=6, end_reg=6):
    
    for codice in range(start_reg, end_reg + 1):
        # nome file input/output
        input_file = f"{JSON_STEP1_FOLDER}{codice:02d}_risultati.json"
        output_file = f"{JSON_STEP2_FOLDER}{codice:02d}.1_risultati.json"

        if not os.path.exists(input_file):
            print(f"⚠️ File non trovato: {input_file}, skip")
            continue

        print(f"\n=== Regione {codice:02d} ===")
        print(f"📥 Input:  {input_file}")
        print(f"📤 Output: {output_file}") 
  
  
    
        # --- Carica input ---
        with open(input_file, "r", encoding="utf-8") as f:
            dati = json.load(f)

        # userai questa struttura fin da subito
        per_file = OrderedDict()  # file_txt -> {"file": file_txt, "entities": [...]}

        for item in dati:
            file_txt = item.get("file", "")
            
            #entities = item.get("risultato", {}).get("entities", [])

            entities = safe_get_entities(item)

            # crea il bucket per il file se non esiste
            per_file.setdefault(file_txt, {"file": file_txt, "entities": []})

            for ent in entities:
                try:
                    # ===== CHIAMATA AL MODELLO =====
                    input_user_message=build_user_message(ent, file_txt)
                    print({input_user_message})
                    resp = client.responses.create(
                        model=OPENAI_MODEL,
                        instructions=SYSTEM_PROMPT,
                        input=input_user_message
                    )
                    output_text = getattr(resp, "output_text", None)
                    if output_text is None:
                        try:
                            output_text = str(resp)
                        except Exception:
                            output_text = ""

                    data, parse_err = extract_json_from_text(output_text)
                    if parse_err or not isinstance(data, dict):
                        print(f"!! Parsing fallito per {file_txt} - entity {ent.get('nome','')}: {parse_err}")
                        # in caso di fallimento, puoi decidere: saltare, oppure inserire un “placeholder” pulito
                        continue

                    # L’output del modello ha forma:
                    # { "file": "<...>", "entities": [ { ENTITA ENRICHED } ] }
                    # Estraggo le entità e le appendo al bucket del file.
                    out_ents = data.get("entities") or []
                    per_file[file_txt]["entities"].extend(out_ents)

                except Exception as e:
                    print(f"❌ Errore con {file_txt} - entity {ent.get('nome', '')}: {e}")
                    continue

        # (Opzionale) De-duplicazione per nome+comune per ciascun file
        for f, bucket in per_file.items():
            seen = set()
            dedup = []
            for ent in bucket["entities"]:
                key = (ent.get("nome","").strip().lower(), ent.get("comune","").strip().lower())
                if key not in seen:
                    seen.add(key)
                    dedup.append(ent)
            bucket["entities"] = dedup

        # Output finale raggruppato
        output_finale = list(per_file.values())
        print(json.dumps(output_finale, ensure_ascii=False, indent=2))

        save_result(output_finale, output_file)
        
main(2, 2)       


=== Regione 02 ===
📥 Input:  output/json/step_1/02_risultati.json
📤 Output: output/json/step_2/02.1_risultati.json
{'FILE: 02_2406_rta_02_240904124958_4166---lrn4del25feb2013.txt\nCODICE_ISTAT_REGIONE: 02\nREGIONE_DA_FILE: Valle D\'Aosta\n\nISTRUZIONI:\nAnalizza il testo fornito ed estrai per ogni entità le chiavi:\n- nome, tipo, comune, indirizzo, ente_capofila, note\n\nPoi applica rigorosamente:\n\n1) Se "indirizzo" è vuoto:\n   a) Se comune è vuoto → cerca un comune valido per quel nome.\n   b) Se "nome" è valorizzato → cerca un comune valido per quel nome. \n   In entrambi i casi → valorizza anche provincia e regione (dedotte se necessario).\n\n2) Se "indirizzo" è valorizzato:\n   - Estrai/deduci "comune"; se differisce → registra conflitto.\n   - Valorizza "provincia" e "regione" (o deducile dal comune).\n   - Non cambiarlo.\n\n3) Normalizza:\n   - Pulisci spazi/virgolette; sigle in MAIUSCOLO.\n   - Uniforma "tipo" alle categorie predefinite. \n   - Normalizza sempre provincia al